# Phase 3a - Input resolution ablation (Colab T4)

Trains each detector at **416 / 640 / 832** with everything else held fixed,
so the only variable is input resolution.

**Budget (measured from the baselines):** ~2.0 h child + ~2.3 h hazard at 30
epochs = **~4.3 h total**. Run **one model per session** — each is comfortably
inside a single Colab session.

**Resumable:** every finished resolution is appended to
`results/metrics/ablation_imgsz_<model>.csv` and skipped on re-run. Runs go to
Drive, so a disconnect costs at most the in-flight resolution.

**Caveat for the write-up:** 30 epochs is a comparison budget, not
convergence. Higher resolutions can need more epochs to pay off, so this may
understate 832. If 832 wins anyway, that's a strong result.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.106 roboflow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ABL_PROJECT = "/content/drive/MyDrive/deeplrn_group2/runs"
import os; os.makedirs(ABL_PROJECT, exist_ok=True)
print("runs ->", ABL_PROJECT)

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
# Both datasets + the two mandatory post-download fixes
!python scripts/download_data.py --child-version 3 --hazard-version 1
!python scripts/fix_data_yaml.py data/child/data.yaml data/hazard/data.yaml
!python scripts/normalize_child_labels.py data/child

## Session A - hazard (~2.3 h)

Uses the Phase 2 winner (`lr0=8.8e-4, box=8.14, cls=0.75, dfl=1.06`) held
fixed at every resolution. Re-run this cell to resume.

In [ ]:
!python scripts/ablation_imgsz.py --model hazard --data data/hazard/data.yaml     --epochs 30 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_imgsz_hazard.zip results/metrics/ablation_imgsz_hazard.csv "$ABL_PROJECT"/ablation_imgsz_hazard_*
!unzip -l ablation_imgsz_hazard.zip | tail -5
from google.colab import files
files.download("ablation_imgsz_hazard.zip")

## Session B - child (~2.0 h)

Child was never tuned, so this runs ultralytics defaults — stated as an
asymmetry in the write-up. Within the model, all three runs are identical
except resolution, so the comparison is still clean.

In [ ]:
!python scripts/ablation_imgsz.py --model child --data data/child/data.yaml     --epochs 30 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_imgsz_child.zip results/metrics/ablation_imgsz_child.csv "$ABL_PROJECT"/ablation_imgsz_child_*
!unzip -l ablation_imgsz_child.zip | tail -5
from google.colab import files
files.download("ablation_imgsz_child.zip")

## Results

In [ ]:
import pandas as pd, os
for m in ("hazard", "child"):
    p = f"results/metrics/ablation_imgsz_{m}.csv"
    if os.path.isfile(p):
        print(f"--- {m} ---")
        print(pd.read_csv(p)[["imgsz","mAP50","mAP50_95","train_min","infer_ms"]]
              .to_string(index=False), "
")
print("baselines @640/100ep: child mAP50=0.947 | hazard mAP50=0.566")

### Notes
- `infer_ms` per resolution feeds the computational-cost comparison the
  proposal promises (two models per frame vs one).
- The two detectors are independent, so they may legitimately end up at
  **different** resolutions (e.g. child 416, hazard 832).
- Val split only; the test split stays untouched until the end.